# 00 — HEAL Algorithm Quick Test

**No CIFAR-10 download. No long training loop.**

Uses tiny synthetic data (random tensors) to verify every algorithmic step fires correctly:

| Step | What it tests |
|---|---|
| 1 | Client local training changes weights |
| 2 | Client sends model to hub (buffering) |
| 3 | Hub FedAvg over client submissions |
| 4 | Hub-to-hub model exchange |
| 5 | Hub broadcasts to clients |
| 6 | Hub failure → self-election in 1 cycle |
| 7 | Full end-to-end cycle produces lower loss |

In [ ]:
import subprocess, sys, os

repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, "src")):
    repo_root = os.path.dirname(repo_root)

assert os.path.isdir(os.path.join(repo_root, "src")), \
    f"src/ not found. cwd={os.getcwd()} — open this notebook from the fl-rockstar repo root"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

subprocess.check_call([sys.executable, "-m", "pip", "install",
    "torch", "torchvision", "datasets", "networkx",
    "numpy", "scipy", "matplotlib", "seaborn", "pandas",
    "tqdm", "pyyaml", "--quiet"])

import copy, random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.model import SimpleCNN
from src.simulation.heal_node import HealClient, HealHub
from src.topology.elevator import build_elevator_topology, heal_recover_hubs
from src.simulation.runner import run_heal_cycle
from src.failures.injectors import inject_crash

device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print(f"device: {device}  |  root: {repo_root}")
print("Imports OK")

In [2]:
# ── Tiny synthetic dataset ────────────────────────────────────────────────
# 8 nodes, 2 hubs, 64 samples per node, 10 classes
# Shape: (N, 3, 32, 32) mimics CIFAR-10

N_NODES = 8
N_HUBS  = 2
N_SAMPLES = 64
SEED = 0

random.seed(SEED)
torch.manual_seed(SEED)

def make_synthetic_loader(n_samples=64, batch_size=16):
    X = torch.randn(n_samples, 3, 32, 32)
    y = torch.randint(0, 10, (n_samples,))
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)

node_data = {i: make_synthetic_loader() for i in range(N_NODES)}
test_loader = make_synthetic_loader(n_samples=128, batch_size=64)

print(f'Synthetic data ready: {N_NODES} nodes, {N_SAMPLES} samples each')

Synthetic data ready: 8 nodes, 64 samples each


In [3]:
# ── Build topology + nodes ────────────────────────────────────────────────
G, hub_ids = build_elevator_topology(N_NODES, N_HUBS, seed=SEED)
print(f'Elevator topology: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}')
print(f'Hub IDs: {hub_ids}')

init_model = SimpleCNN()

hubs    = [HealHub(i,    copy.deepcopy(init_model), node_data[i]) for i in hub_ids]
clients = [HealClient(i, copy.deepcopy(init_model), node_data[i])
           for i in range(N_NODES) if i not in hub_ids]

print(f'Hubs: {len(hubs)} | Clients: {len(clients)}')

Elevator topology: nodes=8, edges=25
Hub IDs: [6, 7]
Hubs: 2 | Clients: 6


In [4]:
# ── STEP 1: Client local training ────────────────────────────────────────
print('STEP 1 — Client local training')

c = clients[0]
w_before = c.model.state_dict()['features.0.weight'].clone()

trained_model = c.train_local(epochs=1)

w_after = c.model.state_dict()['features.0.weight']
changed = not torch.allclose(w_before, w_after)
print(f'  Weights changed after train_local: {changed}')
assert changed, 'ERROR: weights did not change after training!'
print('  PASS')

STEP 1 — Client local training
  Weights changed after train_local: True
  PASS


In [5]:
# ── STEP 2: Client sends model to hub ────────────────────────────────────
print('STEP 2 — Client pushes model to hub buffer')

hub = hubs[0]
assert len(hub._client_models) == 0, 'Hub buffer should start empty'

for c in clients[:3]:  # send 3 client models to hub
    hub.receive_from_client(c.train_local(epochs=1))

print(f'  Hub buffer size: {len(hub._client_models)} (expected 3)')
assert len(hub._client_models) == 3
print('  PASS')

STEP 2 — Client pushes model to hub buffer
  Hub buffer size: 3 (expected 3)
  PASS


In [6]:
# ── STEP 3: Hub FedAvg over client submissions ───────────────────────────
print('STEP 3 — Hub aggregates client models (FedAvg)')

w_hub_before = hub.model.state_dict()['features.0.weight'].clone()
aggregated = hub.aggregate_clients()
w_hub_after = hub.model.state_dict()['features.0.weight']

buffer_cleared = len(hub._client_models) == 0
weights_changed = not torch.allclose(w_hub_before, w_hub_after)

print(f'  Buffer cleared after aggregation: {buffer_cleared}')
print(f'  Hub model updated: {weights_changed}')
assert buffer_cleared, 'ERROR: buffer not cleared after aggregate_clients'
assert weights_changed, 'ERROR: hub weights did not change after FedAvg'
print('  PASS')

STEP 3 — Hub aggregates client models (FedAvg)
  Buffer cleared after aggregation: True
  Hub model updated: True
  PASS


In [7]:
# ── STEP 4: Hub-to-hub exchange ──────────────────────────────────────────
print('STEP 4 — Hub-to-hub model exchange')

# Both hubs aggregate their clients first
for h in hubs:
    for c in clients[:2]:
        h.receive_from_client(c.train_local(epochs=1))
hub_local_models = [h.aggregate_clients() for h in hubs]

# Hub 0 before exchange
w0_before = hubs[0].model.state_dict()['features.0.weight'].clone()

# Hub 0 exchanges with Hub 1
hubs[0].aggregate_hubs([hub_local_models[1]])

w0_after = hubs[0].model.state_dict()['features.0.weight']
changed = not torch.allclose(w0_before, w0_after)
print(f'  Hub 0 model updated after hub exchange: {changed}')
assert changed, 'ERROR: hub-to-hub exchange did not update model'
print('  PASS')

STEP 4 — Hub-to-hub model exchange
  Hub 0 model updated after hub exchange: True
  PASS


In [8]:
# ── STEP 5: Hub broadcasts global model to clients ───────────────────────
print('STEP 5 — Hub broadcasts to clients')

c = clients[0]
w_client_before = c.model.state_dict()['features.0.weight'].clone()

c.receive_global([hubs[0].model, hubs[1].model])

w_client_after = c.model.state_dict()['features.0.weight']
changed = not torch.allclose(w_client_before, w_client_after)
print(f'  Client model updated after receive_global: {changed}')
assert changed, 'ERROR: client did not update after receiving global model'
print('  PASS')

STEP 5 — Hub broadcasts to clients
  Client model updated after receive_global: True
  PASS


In [10]:
# ── STEP 6: Hub failure → self-election ──────────────────────────────────
#
# How "killing" a hub works:
#   1. G.remove_node(hub_id)  → node + all its edges vanish from the graph
#   2. hub_id dropped from hub_ids list → no longer considered a hub
#   3. heal_recover_hubs() finds highest-degree surviving node
#   4. Promotes it: adds edges to ALL remaining nodes → diameter=2 restored
#
# We test heal_recover_hubs() directly (NOT inject_crash) because:
#   inject_crash has a max(1, n_crash) guard → always kills ≥1 node even
#   at crash_rate=0.0, which would silently mutate the graph mid-test.
import networkx as nx

print('STEP 6 — Hub failure + self-election (heal_recover_hubs)')

G2 = copy.deepcopy(G)
hub_ids_copy = list(hub_ids)
failed_hub_id = hub_ids_copy[0]

print(f'  Before: hub_ids={hub_ids_copy}, nodes={G2.number_of_nodes()}, diameter={nx.diameter(G2)}')

G2, recovered_hub_ids = heal_recover_hubs(
    G2, hub_ids_copy, failed_ids=[failed_hub_id], n_hubs=N_HUBS
)

diameter_after = nx.diameter(G2)
print(f'  After:  hub_ids={recovered_hub_ids}, nodes={G2.number_of_nodes()}, diameter={diameter_after}')
print(f'  Hub {failed_hub_id} removed from graph:    {failed_hub_id not in G2.nodes()}')
print(f'  Hub {failed_hub_id} removed from hub_ids:  {failed_hub_id not in recovered_hub_ids}')
print(f'  Hub count maintained at {N_HUBS}:          {len(recovered_hub_ids) == N_HUBS}')
print(f'  Diameter still 2 after re-election:        {diameter_after == 2}')

assert failed_hub_id not in G2.nodes(),       'ERROR: failed hub still in graph'
assert failed_hub_id not in recovered_hub_ids,'ERROR: failed hub still in hub_ids'
assert len(recovered_hub_ids) == N_HUBS,      f'ERROR: expected {N_HUBS} hubs, got {len(recovered_hub_ids)}'
assert diameter_after == 2,                   f'ERROR: diameter={diameter_after}, expected 2'
print('  PASS')

STEP 6 — Hub failure + self-election (heal_recover_hubs)
  Before: hub_ids=[6, 7], nodes=7, diameter=2
  After:  hub_ids=[7, 1], nodes=7, diameter=2
  Hub 6 removed from graph:    True
  Hub 6 removed from hub_ids:  True
  Hub count maintained at 2:          True
  Diameter still 2 after re-election:        True
  PASS


In [11]:
# ── STEP 7: Full cycle end-to-end — loss should decrease ─────────────────
print('STEP 7 — End-to-end: 5 full HEAL cycles, loss should decrease')

# Fresh nodes
torch.manual_seed(SEED)
fresh_model = SimpleCNN()
G_fresh, hub_ids_fresh = build_elevator_topology(N_NODES, N_HUBS, seed=SEED)
hubs_fresh    = [HealHub(i,    copy.deepcopy(fresh_model), node_data[i]) for i in hub_ids_fresh]
clients_fresh = [HealClient(i, copy.deepcopy(fresh_model), node_data[i])
                 for i in range(N_NODES) if i not in hub_ids_fresh]

def avg_loss(nodes, loader):
    criterion = nn.CrossEntropyLoss()
    total_loss, count = 0.0, 0
    for node in nodes:
        node.model.eval()
        with torch.no_grad():
            for X, y in loader:
                total_loss += criterion(node.model(X), y).item()
                count += 1
    return total_loss / count if count else float('inf')

all_nodes = clients_fresh + hubs_fresh
loss_before = avg_loss(all_nodes, test_loader)
print(f'  Loss before training: {loss_before:.4f}')

for cycle in range(5):
    run_heal_cycle(clients_fresh, hubs_fresh, n_sends=1)

loss_after = avg_loss(all_nodes, test_loader)
print(f'  Loss after 5 cycles:  {loss_after:.4f}')
print(f'  Loss decreased: {loss_after < loss_before}')
assert loss_after < loss_before, 'ERROR: loss did not decrease after training!'
print('  PASS')

STEP 7 — End-to-end: 5 full HEAL cycles, loss should decrease
  Loss before training: 2.3007
  Loss after 5 cycles:  2.2993
  Loss decreased: True
  PASS


In [12]:
# ── Summary ───────────────────────────────────────────────────────────────
print()
print('=' * 50)
print('ALL 7 STEPS PASSED')
print('=' * 50)
print()
print('Algorithm verification complete:')
print('  [OK] Step 1 — Client local training')
print('  [OK] Step 2 — Client → Hub buffering')
print('  [OK] Step 3 — Hub FedAvg (aggregate_clients)')
print('  [OK] Step 4 — Hub-to-hub exchange (aggregate_hubs)')
print('  [OK] Step 5 — Hub broadcasts to clients (receive_global)')
print('  [OK] Step 6 — Hub failure → self-election (heal_recover_hubs)')
print('  [OK] Step 7 — End-to-end: 5 cycles reduce loss')
print()
print('Ready to run full experiments → notebooks/03_e1_crash_free.ipynb')


ALL 7 STEPS PASSED

Algorithm verification complete:
  [OK] Step 1 — Client local training
  [OK] Step 2 — Client → Hub buffering
  [OK] Step 3 — Hub FedAvg (aggregate_clients)
  [OK] Step 4 — Hub-to-hub exchange (aggregate_hubs)
  [OK] Step 5 — Hub broadcasts to clients (receive_global)
  [OK] Step 6 — Hub failure → self-election (heal_recover_hubs)
  [OK] Step 7 — End-to-end: 5 cycles reduce loss

Ready to run full experiments → notebooks/03_e1_crash_free.ipynb
